# Music Tagger Android App Builder

This notebook builds the Music Tagger Android APK using Google Colab.

## Instructions

1. Upload this notebook to Google Colab (https://colab.research.google.com/)
2. Run each cell in sequence by clicking the play button or pressing Shift+Enter
3. When prompted, upload your project files
4. Wait for the build to complete
5. Download the generated APK file

## Step 1: Install Dependencies

First, we'll install all the necessary packages to build the Android app.

In [ ]:
%%time
# Install required packages for the build environment
!apt-get update
!apt-get install -y git zip unzip openjdk-8-jdk python3-pip autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev

# Install Buildozer
!pip install Cython==0.29.33
!pip install buildozer==1.5.0

# Install additional Python dependencies
!pip install kivy==2.2.1
!pip install kivymd==1.1.1
!pip install watchdog==3.0.0
!pip install mutagen==1.47.0
!pip install tinytag==2.1.0


## Step 2: Set Up Build Directory

Let's create a directory structure for our build.

In [ ]:
# Create build directory
!mkdir -p music_tagger
%cd music_tagger

## Step 3: Upload Project Files

Now, you'll need to upload your project files. You can either:
1. Upload individual files using the code below, or
2. Create a ZIP file of your project and upload it (recommended for multiple files)

In [ ]:
from google.colab import files

# Option 1: Upload a ZIP file containing the project
uploaded = files.upload()  # Upload your project ZIP file

# Check if a zip file was uploaded and extract it
import os
zip_files = [f for f in uploaded.keys() if f.endswith('.zip')]
if zip_files:
    zip_file = zip_files[0]
    print(f"Extracting {zip_file}...")
    !unzip -o "$zip_file"
    print("Extraction complete!")
else:
    print("No ZIP file uploaded. Please upload project files individually.")

In [ ]:
# Option 2: Upload individual project files if you didn't upload a ZIP
# Uncomment and run this cell if you need to upload files individually

# List of required files for the project
required_files = [
    'music_tagger_mobile.py',  # Main application file
    'buildozer.spec',          # Buildozer configuration
    'music_tagger.py',         # Core functionality
    'config.ini.sample',       # Sample configuration
    'README.md'                # Optional documentation
]

print("Please upload each of the required files when prompted.")
for file in required_files:
    print(f"\nUploading {file}...")
    files.upload()  # This will prompt for a file upload

## Step 4: Ensure Main File is Named main.py

Buildozer looks for main.py as the entry point for the app.

In [ ]:
# If the mobile app file isn't named main.py, copy it to main.py
!cp music_tagger_mobile.py main.py

# List files to verify
!ls -la

## Step 5: Review and Update buildozer.spec

Let's check if the buildozer.spec file exists and create it if needed.

In [ ]:
# Check if buildozer.spec exists
import os

if not os.path.exists('buildozer.spec'):
    print("buildozer.spec not found. Initializing a new one...")
    !buildozer init
    # Update the default spec file for our app
    !sed -i 's/title = My Application/title = Music Tagger/g' buildozer.spec
    !sed -i 's/package.name = myapp/package.name = musictagger/g' buildozer.spec
    !sed -i 's/package.domain = org.test/package.domain = com.januarydecember/g' buildozer.spec
    !sed -i 's/requirements = python3/requirements = python3,kivy==2.2.1,kivymd==1.1.1,watchdog==3.0.0,mutagen==1.47.0,tinytag==2.1.0,pyjnius/g' buildozer.spec
    !sed -i 's/#android.permissions =/android.permissions = INTERNET,READ_EXTERNAL_STORAGE,WRITE_EXTERNAL_STORAGE/g' buildozer.spec
else:
    print("buildozer.spec found!")

# Display the buildozer.spec file
!cat buildozer.spec

## Step 6: Build the Android APK

Now let's build the APK. This process will take about 15-20 minutes.

In [ ]:
%%time
# Build the Android app
!buildozer android debug

## Step 7: Download the APK

If the build was successful, let's download the APK file.

In [ ]:
# Check if the APK was built successfully
import os

apk_path = './bin/musictagger-0.1-debug.apk'
if os.path.exists(apk_path):
    print("APK built successfully!")
    
    # Download the APK file
    from google.colab import files
    files.download(apk_path)
else:
    print("APK build failed. Check the build logs for errors.")
    
    # List all files in bin directory if it exists
    if os.path.exists('./bin'):
        print("\nContents of bin directory:")
        !ls -la ./bin

## Troubleshooting

If you encounter any errors during the build process, here are some common issues and solutions:

1. **Memory errors**: Colab has limited RAM. Try restarting the runtime before building.
2. **Missing dependencies**: Check if all requirements are correctly specified in buildozer.spec.
3. **Java version issues**: The build requires Java 8. If you see Java-related errors, verify the Java version.
4. **Connection timeouts**: Network issues can occur. Try rerunning the build cell.

If you're still having problems, check the buildozer logs at `./.buildozer/logs/`.

In [ ]:
# View buildozer logs if there were errors
!cat ./.buildozer/logs/buildozer.log